In [ ]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

In [ ]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [ ]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [ ]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Control Packet Hierachy

### CQ1.01u - Which MQTT Control Packet has which Fixed Header / Variable Header / Payload?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?type ?fixed ?variable ?payload WHERE {
  ?packet a mqtt:ControlPacket ;
       a ?type.

  OPTIONAL { ?packet mqtt:hasFixedHeader ?fixed }
  OPTIONAL { ?packet mqtt:hasVariableHeader ?variable }
  OPTIONAL { ?packet mqtt:hasPayload ?payload }


  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short